# Traitement des données pour le niveau Silver

## Import des bibliothèques nécessaires

In [ ]:
import pyspark as py

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Création de la session Spark et création de la liste Dataframe

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("olist-silver") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

df_orders = spark.read.parquet("../data/bronze/orders/")
df_customers = spark.read.parquet("../data/bronze/customers/")
df_items = spark.read.parquet("../data/bronze/order_items/")
df_payments = spark.read.parquet("../data/bronze/order_payments/")
df_reviews = spark.read.parquet("../data/bronze/order_reviews/")
df_products = spark.read.parquet("../data/bronze/products/")
df_sellers = spark.read.parquet("../data/bronze/sellers/")
df_geo = spark.read.parquet("../data/bronze/geolocation/")
df_pcnt = spark.read.parquet("../data/bronze/product_category_name_translation/")

dataframes = {
    "orders": df_orders,
    "customers": df_customers,
    "items": df_items,
    "payments": df_payments,
    "reviews": df_reviews,
    "products": df_products,
    "sellers": df_sellers,
    "geo": df_geo,
    "pcnt": df_pcnt
}

## Création de fonctions pour le traitement des données

### Audit des données NULL

In [ ]:
from pyspark.sql.functions import col, sum as _sum, count, when

def audit_nulls(df, name):
    null_counts = df.select([_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])
    null_counts.show()
    print(f"DataFrame: {name}")
    print(f"Total Rows: {df.count()}")
    print(f"Total Columns: {len(df.columns)}")
    print(f"Null Counts: {null_counts.collect()[0].asDict()}")

for df_name, df in dataframes.items():
    audit_nulls(df, df_name)

### Audit des données dupliquées

In [ ]:
def audit_duplicates(df, name):
    duplicate_count = df.count() - df.dropDuplicates().count()
    print(f"DataFrame: {name}")
    print(f"Total Rows: {df.count()}")
    print(f"Duplicate Rows: {duplicate_count}")

for df_name, df in dataframes.items():
    audit_duplicates(df, df_name)

### Conversion des types de données

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType

print("Avant conversion des types de données :")
df_reviews.printSchema()

df_reviews = df_reviews \
    .withColumn("review_score", col("review_score").cast(IntegerType()))

print("Après conversion des types de données :")
df_reviews.printSchema()

## Traitement des données pour le niveau Silver

### Supression des doublons

In [ ]:
df_orders = df_orders.dropna(subset=["order_id", "customer_id"])
df_items = df_items.dropna(subset=["order_id", "product_id"])
df_reviews = df_reviews.dropna(subset=["order_id"])

### Insertion de valeurs par défaut pour les colonnes NULL

In [ ]:
from pyspark.sql.functions import lit

# Texte manquant dans les avis → remplacer par chaîne vide
df_reviews = df_reviews.fillna({
    "review_comment_title": "",
    "review_comment_message": ""
})

# Catégorie produit inconnue
df_products = df_products.fillna({
    "product_category_name": "unknown"
})

# Valeurs numériques manquantes
df_items = df_items.fillna({"freight_value": 0.0})

### Supression des doublons après insertion de valeurs par défaut

In [ ]:
# Sur la clé primaire uniquement
df_orders = df_orders.dropDuplicates(["order_id"])
df_customers = df_customers.dropDuplicates(["customer_id"])
df_products = df_products.dropDuplicates(["product_id"])
df_sellers = df_sellers.dropDuplicates(["seller_id"])

# Géolocalisation : doublons fréquents et attendus → dédoublonner sur le zip
df_geo = df_geo.dropDuplicates(["geolocation_zip_code_prefix"])

## Audit des données après traitement

In [ ]:
from pyspark.sql.functions import col

# Commandes livrées AVANT d'être achetées = incohérent
incoherences = df_orders.filter(
    col("order_delivered_customer_date") < col("order_purchase_timestamp")
)
print(f"Livraisons avant achat : {incoherences.count()}")

# Commandes approuvées avant achat
approved_before_buy = df_orders.filter(
    col("order_approved_at") < col("order_purchase_timestamp")
).count()

print(f"Commandes approuvées avant achat : {approved_before_buy}")

# Articles avec prix <= 0
print(f"Articles avec prix <= 0 : {df_items.filter(col('price') <= 0).count()}")

# Paiements avec valeur < 0
print(f"Paiements avec valeur < 0 : {df_payments.filter(col('payment_value') < 0).count()}")

# Avis avec score < 1 ou > 5
print(f"Avis avec score < 1 ou > 5 : {df_reviews.filter((col('review_score') < 1) | (col('review_score') > 5)).count()}")

## Sauvegarde des DataFrames traités dans le niveau Silver

In [ ]:
df_orders.write.mode("overwrite").parquet("../data/silver/orders/")
df_customers.write.mode("overwrite").parquet("../data/silver/customers/")
df_items.write.mode("overwrite").parquet("../data/silver/order_items/")
df_payments.write.mode("overwrite").parquet("../data/silver/payments/")
df_reviews.write.mode("overwrite").parquet("../data/silver/reviews/")
df_products.write.mode("overwrite").parquet("../data/silver/products/")
df_sellers.write.mode("overwrite").parquet("../data/silver/sellers/")
df_geo.write.mode("overwrite").parquet("../data/silver/geolocation/")
df_pcnt.write.mode("overwrite").parquet("../data/silver/product_category_name_translation/")